In [2]:
import numpy as np
import pandas as pd

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/aadarshjain0307/train-test-data/x_test.parquet
/kaggle/input/datasets/aadarshjain0307/train-test-data/x_train.parquet
/kaggle/input/datasets/aadarshjain0307/train-test-data/x_val.parquet
/kaggle/input/datasets/aadarshjain0307/train-test-data/train_target.parquet
/kaggle/input/datasets/aadarshjain0307/train-test-data/val_target.parquet
/kaggle/input/competitions/rossmann-store-sales/sample_submission.csv
/kaggle/input/competitions/rossmann-store-sales/store.csv
/kaggle/input/competitions/rossmann-store-sales/train.csv
/kaggle/input/competitions/rossmann-store-sales/test.csv


In [4]:
x_train = pd.read_parquet('/kaggle/input/datasets/aadarshjain0307/train-test-data/x_train.parquet')
train_targets = pd.read_parquet('/kaggle/input/datasets/aadarshjain0307/train-test-data/train_target.parquet')
x_val = pd.read_parquet('/kaggle/input/datasets/aadarshjain0307/train-test-data/x_val.parquet')
val_targets = pd.read_parquet('/kaggle/input/datasets/aadarshjain0307/train-test-data/val_target.parquet')
x_test = pd.read_parquet('/kaggle/input/datasets/aadarshjain0307/train-test-data/x_test.parquet')
test_df = pd.read_csv('/kaggle/input/competitions/rossmann-store-sales/test.csv')

In [5]:
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor,StackingRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

In [8]:
model = xgb.XGBRegressor(
    n_estimators = 500,
    learning_rate=0.1,
    max_depth=9,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
)

In [9]:
model.fit(x_train,train_targets)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='logloss',
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=9, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=-1,
             num_parallel_tree=None, ...)

In [10]:
pred = model.predict(x_val)
root_mean_squared_error(pred,val_targets)

1854.2757568359375

In [12]:
base_models = [
    ('rf1',RandomForestRegressor(n_jobs=-1)),
    ('rf2',RandomForestRegressor(n_estimators=200,max_features=0.7,max_samples=0.7,n_jobs=-1)),
    ('xgb1',xgb.XGBRegressor(n_estimators=500,learning_rate=0.1,max_depth=9,n_jobs=-1)),
    ('xgb2',xgb.XGBRegressor(n_jobs=-1))
]

In [13]:
meta_model = xgb.XGBRegressor(n_estimators=200,n_jobs=-1)

In [16]:
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=4,
)

In [17]:
stacking_model.fit(x_train,train_targets)

/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_stacking.py:1060: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


StackingRegressor(cv=4,
                  estimators=[('rf1', RandomForestRegressor(n_jobs=-1)),
                              ('rf2',
                               RandomForestRegressor(max_features=0.7,
                                                     max_samples=0.7,
                                                     n_estimators=200,
                                                     n_jobs=-1)),
                              ('xgb1',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categ...
                                               feature_weights=None, gamma=None,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=None, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None,
                                               max_depth=None, max_leaves=None,
                                               min_child_weight=None,
                                               missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=200, n_jobs=-1,
                                               num_parallel_tree=None, ...))

In [18]:
stack_pred = stacking_model.predict(x_val)
root_mean_squared_error(stack_pred,val_targets)

1376.8458251953125

In [19]:
test_pred = stacking_model.predict(x_test)

In [21]:
submission_df = pd.read_csv('/kaggle/input/competitions/rossmann-store-sales/sample_submission.csv')

In [22]:
submission_df['Sales'] = test_pred* test_df['Open'].fillna(1).astype('float')

In [23]:
submission_df.to_csv('submission.csv',index='None')